# Дообучение Qwen3.5-2B для docai-1c (шаг P21)

LoRA в 16 битах (не QLoRA) поверх `unsloth/Qwen3.5-2B` на SFT-датасете из синтетики проекта: картинка страницы или текстовый слой PDF → JSON черновика документа, ровно как в конвейере `docai extract`. Результат — GGUF Q8_0 и проектор картинок (mmproj) для Ollama под именем `docai-qwen35-2b-ft`.

**Как запускать:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU** → «Сохранить»; затем «Среда выполнения» → **«Выполнить всё»**. Подробно, со временем и действиями при обрыве — `docs/FINETUNE.md`.

**Обрыв сессии не страшен:** чекпоинты пишутся в Google Drive каждые 10 шагов. После обрыва снова «Выполнить всё» — обучение продолжится с последнего чекпоинта; готовый адаптер и готовый GGUF второй раз не считаются.

Ноутбук написан по схеме официального ноутбука Unsloth [Qwen3.5 (2B) Vision](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb) и [руководства Unsloth по Qwen3.5](https://unsloth.ai/docs/models/qwen3.5/fine-tune): те же `FastVisionModel`, 16-битная LoRA, `use_gradient_checkpointing="unsloth"`, `UnslothVisionDataCollator`; команды установки — как в его первой ячейке. Отличия и их причины — в комментариях и в `docs/DECISIONS.md` (D77).

## 1. Настройки

In [ ]:
# Обычно менять ничего не нужно: параметры обучения — в train_config.yaml внутри архива датасета
# (в репозитории — configs/finetune/train_v1.yaml).
DRIVE_DIR = "/content/drive/MyDrive/docai-finetune"  # папка в Google Drive, куда загружен архив
DATASET_NAME = "sft_v1"                               # архив: {DRIVE_DIR}/sft_v1.zip
OVERRIDES = {}  # поверх train_config.yaml, например {"train": {"max_steps": 100, "time_budget_min": 120}}

## 2. GPU и Google Drive
Colab попросит разрешить доступ к Google Drive — разрешите: туда пишутся чекпоинты и готовая модель.

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if gpu.returncode != 0:
    raise SystemExit(
        "Нет GPU. «Среда выполнения» → «Сменить среду выполнения» → T4 GPU, затем «Выполнить всё»."
    )
print("GPU:", gpu.stdout.strip())

from google.colab import drive

drive.mount("/content/drive")
ZIP_PATH = f"{DRIVE_DIR}/{DATASET_NAME}.zip"
if not os.path.exists(ZIP_PATH):
    raise SystemExit(f"Нет {ZIP_PATH}: загрузите архив в Google Drive (docs/FINETUNE.md, шаг 2).")
print("Архив датасета:", ZIP_PATH, f"{os.path.getsize(ZIP_PATH) / 2**20:.0f} МиБ")

## 3. Установка Unsloth
Команды — как в официальном ноутбуке Unsloth для Qwen3.5: Qwen3.5 нужен transformers 5 (со старыми версиями модель не загрузится). Занимает 3–5 минут.

In [ ]:
%%capture install_log
import importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try:
        import numpy, PIL
        _numpy, _pil = f"numpy=={numpy.__version__}", f"pillow=={PIL.__version__}"
    except ImportError:
        _numpy, _pil = "numpy", "pillow"
    !uv pip install -qqq "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
# Версии — в export_info.json рядом с моделью (воспроизводимость).
import importlib.metadata as metadata

VERSIONS = {}
for package in ("unsloth", "unsloth_zoo", "transformers", "trl", "peft", "torch", "accelerate"):
    try:
        VERSIONS[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        VERSIONS[package] = None
print(VERSIONS)
if not (VERSIONS["unsloth"] and VERSIONS["transformers"]):
    print(install_log.stdout[-3000:], install_log.stderr[-3000:])
    raise SystemExit("Установка не удалась — см. вывод выше и docs/FINETUNE.md, «Если что-то пошло не так».")

## 4. Датасет и параметры

In [ ]:
import yaml

LOCAL = "/content/data"
DATA_DIR = f"{LOCAL}/{DATASET_NAME}"
if not os.path.exists(f"{DATA_DIR}/train.jsonl"):
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(LOCAL)


def deep_update(base, extra):
    for key, value in extra.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            deep_update(base[key], value)
        else:
            base[key] = value
    return base


with open(f"{DATA_DIR}/train_config.yaml", encoding="utf-8") as handle:
    CFG = deep_update(yaml.safe_load(handle), OVERRIDES)
with open(f"{DATA_DIR}/dataset_info.json", encoding="utf-8") as handle:
    DATASET_INFO = json.load(handle)

RUN_DIR = f"{DRIVE_DIR}/runs/{CFG['run_name']}"
CKPT_DIR = f"{RUN_DIR}/checkpoints"       # чекпоинты Trainer (LoRA + оптимизатор + номер шага)
ADAPTER_DIR = f"{RUN_DIR}/adapter"        # итоговый адаптер LoRA
NAME = CFG["export"]["name"]
EXPORT_DIR = f"{DRIVE_DIR}/export/{NAME}" # GGUF, Modelfile, export_info.json — это скачать
os.makedirs(CKPT_DIR, exist_ok=True)
MAX_LEN = CFG["model"]["max_seq_length"]


def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            row = json.loads(line)
            for item in row["messages"][0]["content"]:
                if item["type"] == "image":  # путь в архиве → путь на диске Colab
                    item["image"] = f"{DATA_DIR}/{item['image']}"
            rows.append(row)
    return rows


train_rows = read_jsonl(f"{DATA_DIR}/train.jsonl")
val_rows = read_jsonl(f"{DATA_DIR}/val.jsonl")
if CFG["data"].get("drop_text_examples"):
    train_rows = [r for r in train_rows if r["meta"]["kind"] != "text"]
    val_rows = [r for r in val_rows if r["meta"]["kind"] != "text"]
if CFG["data"]["n_train"]:
    train_rows = train_rows[: CFG["data"]["n_train"]]
from collections import Counter

print("train:", len(train_rows), dict(Counter(r["meta"]["kind"] for r in train_rows)))
print("val:  ", len(val_rows), dict(Counter(r["meta"]["kind"] for r in val_rows)))
print(json.dumps(CFG, ensure_ascii=False, indent=1))

## 5. Модель
16-битная LoRA: Unsloth не советует QLoRA (4 бита) для Qwen3.5 — у этих моделей заметнее ошибки квантизации. Энкодер картинок не обучается (`finetune_vision_layers: false`): проектор mmproj остаётся таким же, как у базовой модели в Ollama.

In [ ]:
from unsloth import FastVisionModel  # первым из ML-библиотек: Unsloth патчит transformers и trl
import torch

TRAINED = os.path.exists(f"{ADAPTER_DIR}/adapter_config.json")
if TRAINED:
    print("Обучение уже закончено — беру адаптер из Google Drive:", ADAPTER_DIR)
    model, processor = FastVisionModel.from_pretrained(
        ADAPTER_DIR, load_in_4bit=False, max_seq_length=MAX_LEN
    )
else:
    model, processor = FastVisionModel.from_pretrained(
        CFG["model"]["base"],
        load_in_4bit=CFG["model"]["load_in_4bit"],       # False: 16-битная LoRA
        use_gradient_checkpointing=CFG["train"]["gradient_checkpointing"],  # "unsloth": длинные примеры влезают в 15 ГБ
        max_seq_length=MAX_LEN,
    )
    L = CFG["lora"]
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=L["finetune_vision_layers"],
        finetune_language_layers=L["finetune_language_layers"],
        finetune_attention_modules=L["finetune_attention_modules"],
        finetune_mlp_modules=L["finetune_mlp_modules"],
        r=L["r"],
        lora_alpha=L["alpha"],
        lora_dropout=L["dropout"],
        bias="none",
        random_state=CFG["seed"],
        use_rslora=False,
    )
TOKENIZER = getattr(processor, "tokenizer", processor)

## 6. Длина примеров
Считается точно, процессором модели (с токенами картинки). Примеры длиннее `max_seq_length` отбрасываются, а не обрезаются: обрезанный JSON учил бы модель обрывать ответ. Длины кэшируются в Drive.

In [ ]:
from PIL import Image


def images_of(row):
    return [
        Image.open(item["image"]).convert("RGB")
        for item in row["messages"][0]["content"]
        if item["type"] == "image"
    ]


def seq_len(row):
    text = processor.apply_chat_template(row["messages"], tokenize=False)
    encoded = processor(text=[text], images=images_of(row) or None)
    return len(encoded["input_ids"][0])


LEN_CACHE = f"{RUN_DIR}/lengths.json"
lengths = json.load(open(LEN_CACHE)) if os.path.exists(LEN_CACHE) else {}
if not TRAINED:
    started = time.time()
    for row in train_rows + val_rows:
        if row["id"] not in lengths:
            lengths[row["id"]] = seq_len(row)
    json.dump(lengths, open(LEN_CACHE, "w"))
    long = [r["id"] for r in train_rows if lengths[r["id"]] > MAX_LEN]
    train_rows = [r for r in train_rows if lengths[r["id"]] <= MAX_LEN]
    values = sorted(lengths[r["id"]] for r in train_rows)
    print(f"посчитано за {time.time() - started:.0f} с; отброшено длинных: {len(long)} {long[:5]}")
    print(
        f"токенов в примере: среднее {sum(values) / len(values):.0f}, "
        f"95% {values[int(len(values) * 0.95)]}, макс {values[-1]}; всего {sum(values):,}"
    )
    print("Пример, как его видит модель (начало и конец):")
    sample = processor.apply_chat_template(train_rows[0]["messages"], tokenize=False)
    print(sample[:300], "\n…\n", sample[-400:])

## 7. Проверка коллатора до обучения
Loss считается только по ответу модели (`train_on_responses_only`), картинки не уменьшаются (`resize="max"`): в датасете они уже 1280 px по длинной стороне, как в конвейере, и Ollama кодирует такую страницу в те же 1120 токенов. По умолчанию коллатор Unsloth сжал бы страницу до `image_size` энкодера (768 px) — мелкий текст документа стал бы нечитаемым.

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator

collator = UnslothVisionDataCollator(
    model,
    processor,
    resize="max",                      # не уменьшать картинки (см. выше)
    max_seq_length=MAX_LEN,
    train_on_responses_only=CFG["data"]["train_only_on_responses"],
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
if not TRAINED:
    probes = [next(r for r in train_rows if r["meta"]["kind"] != "text")]
    probes += [r for r in train_rows if r["meta"]["kind"] == "text"][:1]  # пример без картинки
    for row in probes:
        batch = collator([{"messages": row["messages"]}])
        ids, labels = batch["input_ids"][0], batch["labels"][0]
        answer = TOKENIZER.decode(ids[labels != -100])
        target = row["messages"][1]["content"][0]["text"]
        assert target in answer, "ответ модели в метках не совпал с эталоном"
        if abs(len(ids) - lengths[row["id"]]) > 2:
            print(f"Внимание: коллатор дал {len(ids)} токенов, расчёт длины — {lengths[row['id']]}")
        with torch.no_grad():
            loss = model(**{k: v.to(model.device) for k, v in batch.items()}).loss.item()
        print(
            f"{row['id']}: токенов {len(ids)}, в loss {int((labels != -100).sum())}, "
            f"loss до обучения {loss:.3f}; ответ начинается с {answer[:40]!r}"
        )

## 8. Обучение
Печатает скорость и прогноз после первых шагов. Если прогноз не укладывается в сессию — не страшно: по истечении `time_budget_min` ноутбук сам сохранит чекпоинт и остановится; продолжение — снова «Выполнить всё».

In [ ]:
from transformers import TrainerCallback
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTConfig, SFTTrainer


class TimeBudget(TrainerCallback):
    """Скорость и прогноз; по истечении бюджета сессии — чекпоинт и остановка."""

    def __init__(self, minutes):
        self.limit = minutes * 60

    def on_train_begin(self, args, state, control, **kwargs):
        self.started, self.first_step = time.time(), state.global_step

    def on_step_end(self, args, state, control, **kwargs):
        done = state.global_step - self.first_step
        elapsed = time.time() - self.started
        if done in (1, 3, 5, 10) or state.global_step % 10 == 0:
            per_step = elapsed / max(done, 1)
            left = (state.max_steps - state.global_step) * per_step
            print(
                f"шаг {state.global_step}/{state.max_steps}: {per_step:.0f} с/шаг, "
                f"до конца ≈ {left / 60:.0f} мин, в этой сессии {elapsed / 60:.0f} мин"
            )
        if elapsed > self.limit:
            print("Бюджет времени сессии исчерпан: сохраняю чекпоинт и останавливаюсь.")
            control.should_save = True
            control.should_training_stop = True
        return control


if not TRAINED:
    T = CFG["train"]
    FastVisionModel.for_training(model)
    eval_rows = [r for r in val_rows if lengths[r["id"]] <= MAX_LEN][: T["eval_examples"]]
    trainer = SFTTrainer(
        model=model,
        tokenizer=processor,
        data_collator=collator,
        train_dataset=[{"messages": r["messages"]} for r in train_rows],
        eval_dataset=[{"messages": r["messages"]} for r in eval_rows],
        callbacks=[TimeBudget(T["time_budget_min"])],
        args=SFTConfig(
            output_dir=CKPT_DIR,                     # прямо в Google Drive
            per_device_train_batch_size=T["batch_size"],
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=T["grad_accum"],
            max_steps=T["max_steps"],
            warmup_steps=T["warmup_steps"],
            learning_rate=T["learning_rate"],
            lr_scheduler_type=T["lr_scheduler"],
            optim=T["optim"],
            weight_decay=T["weight_decay"],
            logging_steps=1,
            save_strategy="steps",
            save_steps=T["save_steps"],
            save_total_limit=T["save_total_limit"],
            eval_strategy="steps",
            eval_steps=T["eval_steps"],
            seed=CFG["seed"],
            data_seed=CFG["seed"],
            report_to="none",
            # Обязательно для обучения на картинках с Unsloth:
            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},
            max_length=MAX_LEN,
        ),
    )
    last = get_last_checkpoint(CKPT_DIR)
    print("Продолжаю с чекпоинта:", last) if last else print("Начинаю обучение с нуля")
    trainer.train(resume_from_checkpoint=last)
    with open(f"{RUN_DIR}/log_history.json", "w", encoding="utf-8") as handle:
        json.dump(trainer.state.log_history, handle, ensure_ascii=False, indent=1)
    if trainer.state.global_step < trainer.state.max_steps:
        raise SystemExit(
            f"Остановлено на шаге {trainer.state.global_step} из {trainer.state.max_steps}: "
            "кончился бюджет времени сессии. Прогресс в Google Drive. Чтобы продолжить: "
            "«Среда выполнения» → «Выполнить всё» (можно в новой сессии или завтра)."
        )
    model.save_pretrained(ADAPTER_DIR)
    processor.save_pretrained(ADAPTER_DIR)
    TRAINED = True
    print("Адаптер LoRA сохранён:", ADAPTER_DIR)

## 9. Быстрая проверка ответа
Три примера val (текст, скан, фото) — модель должна вернуть валидный JSON. Полная оценка — локально (`docs/FINETUNE.md`, шаг 6).

In [ ]:
FastVisionModel.for_inference(model)


def answer(row, max_new_tokens=3000):
    messages = [row["messages"][0]]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=[prompt], images=images_of(row) or None, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    return TOKENIZER.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


QUICK = []
for kind in ("text", "scan", "photo"):
    rows_of_kind = [r for r in val_rows if r["meta"]["kind"] == kind]
    if not rows_of_kind:
        continue
    row = min(rows_of_kind, key=lambda r: r["meta"]["target_chars"])
    started = time.time()
    text = answer(row)
    try:
        parsed, valid = json.loads(text), True
    except json.JSONDecodeError:
        parsed, valid = None, False
    exact = parsed == json.loads(row["messages"][1]["content"][0]["text"]) if valid else False
    QUICK.append({"id": row["id"], "valid_json": valid, "exact": exact, "seconds": round(time.time() - started, 1)})
    print(QUICK[-1], text[:200])

## 10. Слияние LoRA и экспорт в GGUF
Адаптер вливается в 16-битные веса, затем конвертер llama.cpp делает два файла: модель Q8_0 и проектор картинок mmproj (F16). Это тот же конвертер и тот же формат, что у официальной `qwen3.5:2b-q8_0` в Ollama. Конвертер ставится в отдельное окружение, чтобы не сломать библиотеки обучения. Около 15 минут.

In [ ]:
GGUF_NAME = f"{NAME}.Q8_0.gguf"
MMPROJ_NAME = f"{NAME}.mmproj-f16.gguf"
EXPORTED = all(os.path.exists(f"{EXPORT_DIR}/{n}") for n in (GGUF_NAME, MMPROJ_NAME, "Modelfile"))
MERGED = "/content/merged"
WORK = "/content/export"


def run(cmd, **kw):
    print("$", " ".join(cmd))
    done = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if done.returncode != 0:
        print(done.stdout[-4000:], done.stderr[-4000:])
        raise SystemExit(f"Команда завершилась с ошибкой {done.returncode}")
    return done.stdout


if EXPORTED:
    print("GGUF уже в Google Drive:", EXPORT_DIR)
else:
    if not os.path.exists(f"{MERGED}/config.json"):
        model.save_pretrained_merged(MERGED, processor, save_method="merged_16bit")
    if not os.path.exists("/content/llama.cpp"):
        run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", "/content/llama.cpp"])
    LLAMA_CPP_COMMIT = run(["git", "-C", "/content/llama.cpp", "rev-parse", "HEAD"]).strip()
    VENV_PY = "/content/gguf-venv/bin/python"
    if not os.path.exists(VENV_PY):
        run(["uv", "venv", "-q", "/content/gguf-venv", "--python", "3.12"])
        run(["uv", "pip", "install", "-q", "--python", VENV_PY, "-r",
             "/content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt"])
    os.makedirs(WORK, exist_ok=True)
    convert = [VENV_PY, "/content/llama.cpp/convert_hf_to_gguf.py", MERGED]
    run(convert + ["--outtype", CFG["export"]["model_outtype"], "--outfile", f"{WORK}/{GGUF_NAME}"])
    run(convert + ["--mmproj", "--outtype", CFG["export"]["mmproj_outtype"], "--outfile", f"{WORK}/{MMPROJ_NAME}"])
    if not os.path.exists(f"{WORK}/{MMPROJ_NAME}"):  # если конвертер сам добавил префикс к имени
        found = glob.glob(f"{WORK}/*mmproj*.gguf")
        assert len(found) == 1, found
        shutil.move(found[0], f"{WORK}/{MMPROJ_NAME}")
    print(os.listdir(WORK))

## 11. Проверка GGUF
Сверяем заголовки с официальной `qwen3.5:2b-q8_0` из Ollama: архитектура `qwen35`, квантизация Q8_0 и 320 тензоров у модели; у проектора — `clip`, `qwen3vl_merger` и 298 тензоров.

In [ ]:
CHECK = r"""
import json, sys
from gguf import GGUFReader
out = {}
for path in sys.argv[1:]:
    r = GGUFReader(path)
    f = r.fields
    get = lambda k: f[k].contents() if k in f else None
    out[path.rsplit("/", 1)[-1]] = {
        "architecture": get("general.architecture"),
        "file_type": get("general.file_type"),
        "projector_type": get("clip.projector_type"),
        "tensors": len(r.tensors),
    }
print(json.dumps(out))
"""
if not EXPORTED:
    GGUF_INFO = json.loads(run([VENV_PY, "-c", CHECK, f"{WORK}/{GGUF_NAME}", f"{WORK}/{MMPROJ_NAME}"]))
    print(json.dumps(GGUF_INFO, indent=1))
    main, proj = GGUF_INFO[GGUF_NAME], GGUF_INFO[MMPROJ_NAME]
    assert main["architecture"] == "qwen35" and main["file_type"] == 7, main  # 7 = Q8_0
    assert proj["architecture"] == "clip" and proj["projector_type"] == "qwen3vl_merger", proj
    if (main["tensors"], proj["tensors"]) != (320, 298):
        print("Внимание: число тензоров отличается от официальной модели Ollama (320 и 298).")

## 12. Сохранение в Google Drive
В папку `export/docai-qwen35-2b-ft` кладутся два файла GGUF, `Modelfile` для Ollama и `export_info.json` (параметры, версии, результат обучения). Эту папку нужно скачать (`docs/FINETUNE.md`, шаг 5).

In [ ]:
MODELFILE = f"""# {NAME}: Qwen3.5-2B + LoRA на синтетике docai-1c (шаг P21). GGUF Q8_0 + проектор картинок.
# Подключение: scripts/finetune_install.ps1 (или: ollama create {NAME} -f Modelfile в этой папке).
FROM ./{GGUF_NAME}
FROM ./{MMPROJ_NAME}
TEMPLATE {{{{ .Prompt }}}}
RENDERER qwen3.5
PARSER qwen3.5
PARAMETER temperature 0
PARAMETER top_k 20
PARAMETER top_p 0.95
PARAMETER presence_penalty 0
PARAMETER num_ctx 8192
LICENSE \"\"\"Apache-2.0: базовая модель Qwen3.5-2B, https://huggingface.co/Qwen/Qwen3.5-2B\"\"\"
"""
if not EXPORTED:
    os.makedirs(EXPORT_DIR, exist_ok=True)
    logs = json.load(open(f"{RUN_DIR}/log_history.json", encoding="utf-8")) if os.path.exists(f"{RUN_DIR}/log_history.json") else []
    train_losses = [x["loss"] for x in logs if "loss" in x]
    eval_losses = [(x["step"], x["eval_loss"]) for x in logs if "eval_loss" in x]
    import hashlib

    def sha256(path):
        digest = hashlib.sha256()
        with open(path, "rb") as handle:
            for chunk in iter(lambda: handle.read(1 << 24), b""):
                digest.update(chunk)
        return digest.hexdigest()

    info = {
        "name": NAME,
        "sha256": {n: sha256(f"{WORK}/{n}") for n in (GGUF_NAME, MMPROJ_NAME)},  # проверка скачанных файлов
        "base_model": CFG["model"]["base"],
        "train_config": CFG,
        "dataset": {"name": DATASET_NAME, "git_commit": DATASET_INFO.get("git_commit"), "counts": DATASET_INFO.get("counts")},
        "train_examples_used": len(train_rows),
        "steps": max((x.get("step", 0) for x in logs), default=None),
        "train_loss_first_last": [train_losses[0], train_losses[-1]] if train_losses else None,
        "eval_loss": eval_losses,
        "quick_check": QUICK,
        "versions": VERSIONS,
        "llama_cpp_commit": LLAMA_CPP_COMMIT,
        "gguf": GGUF_INFO,
        "gpu": gpu.stdout.strip(),
    }
    with open(f"{WORK}/export_info.json", "w", encoding="utf-8") as handle:
        json.dump(info, handle, ensure_ascii=False, indent=1)
    with open(f"{WORK}/Modelfile", "w", encoding="utf-8", newline="\n") as handle:
        handle.write(MODELFILE)
    for name in ("export_info.json", GGUF_NAME, MMPROJ_NAME, "Modelfile"):  # Modelfile последним: признак, что всё скопировано
        print("копирую в Drive:", name)
        shutil.copyfile(f"{WORK}/{name}", f"{EXPORT_DIR}/{name}")
for name in sorted(os.listdir(EXPORT_DIR)):
    print(f"{name:40s} {os.path.getsize(f'{EXPORT_DIR}/{name}') / 2**20:8.0f} МиБ")
print("\nГотово. Скачайте папку", EXPORT_DIR, "и выполните docs/FINETUNE.md, шаг 5.")